# Predicting metastasis and metastatic organotropism in malignancies using machine learning

**MSc in Biomedical and Health Informatics — Mahidol University, 2026.**
Author: Ching To Chung.

This is the analysis notebook behind the thesis, cleaned for publication. It runs three tasks on the
public MSK-IMPACT 2017 cohort:

1. **Binary** — is a sample a primary tumour or a metastasis? (logistic regression, XGBoost)
2. **Forward organotropism** — given the primary site and mutation profile, which organ is the
   metastasis in? (logistic regression, XGBoost, random forest, SVM, MLP, TabTransformer)
3. **Reverse** — given the metastatic site and mutation profile, which primary site did it come from?
   Same model suite, plus a Bernoulli naive Bayes baseline.

Each task is preceded by its own data loading, so the sections can be read independently — but run the
notebook top to bottom, because helpers and encoders carry over between cells. Outputs have been
cleared: the numbers in the README come from the 2026 run recorded in the thesis, not from a fresh
execution.

Data is not included. See `data/README.md`.


## Setup

In [ ]:
# --- Paths -----------------------------------------------------------------
# Put the two MSK-IMPACT files in ./data, or point MSK_DATA_DIR at your download.
import os
from pathlib import Path

DATA_DIR = Path(os.environ.get("MSK_DATA_DIR", "data"))
OUTPUT_DIR = Path(os.environ.get("MSK_OUTPUT_DIR", "outputs"))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CLINICAL_TSV = DATA_DIR / "msk_impact_2017_clinical_data.tsv"
MUTATIONS_MAF = DATA_DIR / "data_mutations.txt"

print("clinical :", CLINICAL_TSV, "exists:", CLINICAL_TSV.exists())
print("mutations:", MUTATIONS_MAF, "exists:", MUTATIONS_MAF.exists())

## Binary Prediction of Metastasis

### Importing and Merging Data

In [ ]:
import pandas as pd
df=pd.read_csv(CLINICAL_TSV, sep="\t")


# === 1. Load the mutation file (MAF format) ===
# Adjust path to where you saved data_mutations.txt
mutations = pd.read_csv(MUTATIONS_MAF, sep="\t", comment='#', low_memory=False)

print("File shape:", mutations.shape)
print("Columns:", mutations.columns.tolist())
print(mutations.head())

# === 2. Inspect key fields ===
# Usually, 'Hugo_Symbol' = gene, 'Tumor_Sample_Barcode' = sample ID
print(mutations[['Hugo_Symbol', 'Tumor_Sample_Barcode', 'Variant_Classification']].head())

# === 3. Create a binary mutation matrix ===
# Pivot so each gene is a column, each sample is a row
# Mark 1 if mutated, 0 otherwise
binary_matrix = (
    mutations
    .assign(value=1)
    .pivot_table(index='Tumor_Sample_Barcode',
                 columns='Hugo_Symbol',
                 values='value',
                 aggfunc='max',
                 fill_value=0)
)

print("Binary matrix shape:", binary_matrix.shape)
print(binary_matrix.iloc[0:5, 0:10])  # show first 5 samples × first 10 genes

df = df.rename(columns={'Sample ID': 'Tumor_Sample_Barcode'})

merged = df.merge(binary_matrix,
                  how='inner',
                  on='Tumor_Sample_Barcode')

In [ ]:
target_col = "Sample Type"   # binary target
y = (merged[target_col] == "Metastasis").astype(int).values  # 1 = Metastasis, 0 = Primary
categorical_cols = ["Primary Tumor Site", "Smoking History"] + list(merged.iloc[:,27:].columns)
numeric_cols = ["Mutation Count"]

X_categ = merged[categorical_cols].astype(str)
X_num   = merged[numeric_cols]

### Preprocessing and Train/Test Split

In [ ]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    pd.concat([X_categ, X_num], axis=1), y, test_size=0.2, random_state=42, stratify=y
)

# Column transformer
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
        ("num", StandardScaler(), numeric_cols)
    ]
)

### Logistic Regression

In [ ]:
# Logistic Regression pipeline
log_reg = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000, class_weight="balanced"))
])

# Train
log_reg.fit(X_train, y_train)

# Predictions
y_pred = log_reg.predict(X_test)
y_prob = log_reg.predict_proba(X_test)[:, 1]

# Metrics
print("Accuracy:", accuracy_score(y_test, y_pred))
print("F1 (weighted):", f1_score(y_test, y_pred, average="weighted"))
print("ROC AUC:", roc_auc_score(y_test, y_prob))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc

# ROC curve
fpr, tpr, thresholds = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)

# Plot
plt.figure(figsize=(6,6))
plt.plot(fpr, tpr, color="darkorange", lw=2, label=f"ROC curve (AUC = {roc_auc:.2f})")
plt.plot([0, 1], [0, 1], color="navy", lw=2, linestyle="--")
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - Logistic Regression (Metastasis Prediction)")
plt.legend(loc="lower right")
plt.show()

### XGBoost

In [ ]:
from xgboost import XGBClassifier

xgb = XGBClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="auc",
    scale_pos_weight=len(y_train[y_train==0]) / len(y_train[y_train==1])  # balance classes
)

xgb.fit(preprocessor.fit_transform(X_train), y_train)
y_pred = xgb.predict(preprocessor.transform(X_test))
y_prob = xgb.predict_proba(preprocessor.transform(X_test))[:, 1]
# Metrics
print("Accuracy:", accuracy_score(y_test, y_pred))
print("F1 (weighted):", f1_score(y_test, y_pred, average="weighted"))
print("ROC AUC:", roc_auc_score(y_test, y_prob))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc

# ROC curve
fpr, tpr, thresholds = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)

# Plot
plt.figure(figsize=(6,6))
plt.plot(fpr, tpr, color="darkgreen", lw=2, label=f"ROC curve (AUC = {roc_auc:.2f})")
plt.plot([0, 1], [0, 1], color="navy", lw=2, linestyle="--")
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - XGBoost (Metastasis Prediction)")
plt.legend(loc="lower right")
plt.show()

### SHAP

In [ ]:
# Get feature names from the preprocessor
feature_names = preprocessor.get_feature_names_out()
import shap

# Use TreeExplainer for XGB
explainer = shap.TreeExplainer(xgb)

# Transform X_test through preprocessor
X_test_transformed = preprocessor.transform(X_test)

# Compute SHAP values
shap_values = explainer.shap_values(X_test_transformed)

# Summary plot (global importance)
shap.summary_plot(shap_values, X_test_transformed, feature_names=feature_names)

In [ ]:
import numpy as np
import pandas as pd

# Mean absolute SHAP value per feature
mean_abs_shap = np.abs(shap_values).mean(axis=0)

# Put into dataframe
shap_importance = pd.DataFrame({
    "feature": feature_names,
    "importance": mean_abs_shap
}).sort_values(by="importance", ascending=False)

topN = 20
print(shap_importance.head(topN))

import matplotlib.pyplot as plt

shap_importance.head(topN).plot(
    kind="barh", x="feature", y="importance", legend=False, figsize=(8,6)
)
plt.gca().invert_yaxis()
plt.title(f"Top {topN} Features by SHAP Importance (Mean Absolute)")
plt.xlabel("Mean |SHAP value|")
plt.show()

## Predicting Organotropism

### Importing and Merging Data

In [ ]:
import pandas as pd
df=pd.read_csv(CLINICAL_TSV, sep="\t")
df.head()

In [ ]:
import pandas as pd

# === 1. Load the mutation file (MAF format) ===
# Adjust path to where you saved data_mutations.txt
mutations = pd.read_csv(MUTATIONS_MAF, sep="\t", comment='#', low_memory=False)

print("File shape:", mutations.shape)
print("Columns:", mutations.columns.tolist())
print(mutations.head())

# === 2. Inspect key fields ===
# Usually, 'Hugo_Symbol' = gene, 'Tumor_Sample_Barcode' = sample ID
print(mutations[['Hugo_Symbol', 'Tumor_Sample_Barcode', 'Variant_Classification']].head())

# === 3. Create a binary mutation matrix ===
# Pivot so each gene is a column, each sample is a row
# Mark 1 if mutated, 0 otherwise
binary_matrix = (
    mutations
    .assign(value=1)
    .pivot_table(index='Tumor_Sample_Barcode',
                 columns='Hugo_Symbol',
                 values='value',
                 aggfunc='max',
                 fill_value=0)
)

print("Binary matrix shape:", binary_matrix.shape)
print(binary_matrix.iloc[0:5, 0:10])  # show first 5 samples × first 10 genes

# === 4. Save the binary matrix ===
binary_matrix.to_csv(OUTPUT_DIR / "mutation_binary_matrix.csv")


In [ ]:
#Merge binary mutation matrix with clinical data
df = df.rename(columns={'Sample ID': 'Tumor_Sample_Barcode'})

merged = df.merge(binary_matrix,
                  how='inner',
                  on='Tumor_Sample_Barcode')

print("Merged shape:", merged.shape)
print("Columns:", merged.columns[:15])  # show first 15 columns

In [ ]:
merged.to_csv(OUTPUT_DIR / "merged_mutation_clinical.csv", index=False)

### Pre-processing

In [ ]:
#Building a logistic regression model
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

# Drop samples without metastatic site annotation
data = merged.dropna(subset=['Metastatic Site']).copy()

# Define y
y = data['Metastatic Site']


print("Number of metastatic cases:", len(y))
print("Number of classes:", y.nunique())
print("Class distribution:\n", y.value_counts())

# Plot a count bar chart for y
plt.figure(figsize=(12, 8))
y.value_counts().plot(kind='bar')
plt.title('Count of Metastatic Sites')
plt.xlabel('Metastatic Site')
plt.ylabel('Count')
plt.xticks(rotation=90)
plt.show()

In [ ]:
# Merge rare classes into "Other"
class_counts = y.value_counts()
rare_classes = class_counts[class_counts < 20].index   # threshold can be 2, 5, etc.
data['Metastatic Site'] = data['Metastatic Site'].replace(rare_classes, 'Other')

# Redefine y after replacement
y = data['Metastatic Site']
print("Number of classes:", y.nunique())
print("Class distribution:\n", y.value_counts())
# Plot a count bar chart for y
plt.figure(figsize=(12, 8))
y.value_counts().plot(kind='bar')
plt.title('Count of Metastatic Sites')
plt.xlabel('Metastatic Site')
plt.ylabel('Count')
plt.xticks(rotation=90)
plt.show()

In [ ]:
#(1) Primary Site Only
# One-hot encode the categorical "Primary Tumor Site"

primary_encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
X_primary = primary_encoder.fit_transform(data[['Primary Tumor Site']])

# Automatically fetch feature names
primary_feature_names = primary_encoder.get_feature_names_out()
X_primary = pd.DataFrame(X_primary, columns=primary_feature_names, index=data.index)

In [ ]:
#(2) Genetic Mutations Only
# Identify gene columns = all mutation binary features
gene_columns = [col for col in merged.columns if col not in df.columns]  # assuming df is clinical data
X_mut = data[gene_columns].copy()

In [ ]:
#(3) Combined
# Concatenate both primary site and mutation features
X_combined = pd.concat([X_primary, X_mut], axis=1)


### Train / Test Split

In [ ]:
# Use stratified split to preserve distribution of metastatic sites
X_train_p, X_test_p, y_train, y_test = train_test_split(
    X_primary, y, test_size=0.2, random_state=42, stratify=y
)

X_train_m, X_test_m, _, _ = train_test_split(
    X_mut, y, test_size=0.2, random_state=42, stratify=y
)

X_train_c, X_test_c, _, _ = train_test_split(
    X_combined, y, test_size=0.2, random_state=42, stratify=y
)

### Logistic Regression (Multiclass)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report, confusion_matrix
)
from sklearn.preprocessing import LabelEncoder
import numpy as np
import pandas as pd

def evaluate_model(clf, X_train, X_test, y_train, y_test, model_name="Model"):
    # Encode labels (because XGBoost requires numeric targets)
    le = LabelEncoder()
    y_train_enc = le.fit_transform(y_train)
    y_test_enc = le.transform(y_test)

    # Fit on encoded labels
    clf.fit(X_train, y_train_enc)
    y_pred_enc = clf.predict(X_test)

    # Decode predictions back to original string labels
    y_pred = le.inverse_transform(y_pred_enc)

    # Metrics (using original string labels for readability)
    acc = accuracy_score(y_test, y_pred)
    macro_f1 = f1_score(y_test, y_pred, average="macro")
    weighted_f1 = f1_score(y_test, y_pred, average="weighted")
    nir = np.max(np.bincount(y_test.factorize()[0])) / len(y_test)

    # Print summary
    print(f"\n=== {model_name} Summary ===")
    print(f"Accuracy: {acc:.3f}")
    print(f"Macro F1: {macro_f1:.3f}")
    print(f"Weighted F1: {weighted_f1:.3f}")
    print(f"No Information Rate: {nir:.3f}")
    print(f"Better than baseline? {'YES' if acc > nir else 'NO'}")
    print("\nClassification Report:\n", classification_report(y_test, y_pred))

    # Confusion matrix heatmap
    cm = confusion_matrix(y_test, y_pred, labels=le.classes_)
    plt.figure(figsize=(10,8))
    sns.heatmap(cm, cmap="Blues", cbar=False,
                xticklabels=le.classes_,
                yticklabels=le.classes_)
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title(f"Confusion Matrix - {model_name}")
    plt.show()

    # Return results for barplot summary
    return {
        "Model": model_name,
        "Accuracy": acc,
        "MacroF1": macro_f1,
        "WeightedF1": weighted_f1,
        "NIR": nir,
        "TrainedModel": clf,
        "LabelEncoder": le
    }

In [ ]:
from sklearn.linear_model import LogisticRegression

results = []

# Primary only
clf = LogisticRegression(max_iter=500,  
        solver='saga',
        n_jobs=-1)
results.append(evaluate_model(clf, X_train_p, X_test_p, y_train, y_test, "Primary Only"))

# Mutations only
clf = LogisticRegression(max_iter=500, 
        solver='saga',
        n_jobs=-1)
results.append(evaluate_model(clf, X_train_m, X_test_m, y_train, y_test, "Mutations Only"))

# Combined
clf = LogisticRegression(max_iter=500, 
        solver='saga',      # handles multinomial + sparse data
        n_jobs=-1)
results.append(evaluate_model(clf, X_train_c, X_test_c, y_train, y_test, "Combined"))

In [ ]:
df_results = pd.DataFrame(results)

df_results.plot(x="Model", y=["Accuracy", "MacroF1", "WeightedF1"],
                kind="bar", figsize=(8,6))
plt.title("Performance Comparison Across Feature Sets")
plt.ylabel("Score")
plt.xticks(rotation=0)
plt.ylim(0, 1)  # metrics are between 0 and 1
plt.show()

### XGBoost

In [ ]:
from xgboost import XGBClassifier
results = []

res_primary = evaluate_model(
    XGBClassifier(
        objective="multi:softmax",
        num_class=len(np.unique(y_train)),
        eval_metric="mlogloss",
        random_state=42
    ),
    X_train_p, X_test_p, y_train, y_test,
    model_name="XGBoost - Primary Only"
)
results.append(res_primary)

res_mutations = evaluate_model(
    XGBClassifier(
        objective="multi:softmax",
        num_class=len(np.unique(y_train)),
        eval_metric="mlogloss",
        random_state=42
    ),
    X_train_m, X_test_m, y_train, y_test,
    model_name="XGBoost - Mutations Only"
)
results.append(res_mutations)

res_combined = evaluate_model(
    XGBClassifier(
        objective="multi:softmax",
        num_class=len(np.unique(y_train)),
        eval_metric="mlogloss",
        random_state=42
    ),
    X_train_c, X_test_c, y_train, y_test,
    model_name="XGBoost - Combined"
)
results.append(res_combined)

In [ ]:
df_results = pd.DataFrame(results)

df_results.plot(x="Model", y=["Accuracy", "MacroF1", "WeightedF1"],
                kind="bar", figsize=(8,6))
plt.title("Performance Comparison Across Feature Sets")
plt.ylabel("Score")
plt.xticks(rotation=0)
plt.ylim(0, 1)  # metrics are between 0 and 1
plt.show()

### SHAP

In [ ]:
import shap
import matplotlib.pyplot as plt
import os

model = res_combined["TrainedModel"]
le = res_combined["LabelEncoder"]

# Use the test set for SHAP
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test_c)

# Global importance (bar plot)
plt.figure(figsize=(14, 10))
shap.summary_plot(shap_values, X_test_c, plot_type="bar", class_names=le.classes_,max_display=20, show=False)
ax = plt.gca()
for lbl in ax.get_yticklabels():
    lbl.set_fontsize(7)      # or 6–8
    lbl.set_va('center')

ax = plt.gca()

# Shrink x-axis tick labels
ax.tick_params(axis='x', labelsize=8)   # try 6–9

# Shrink x-axis title (the long SHAP caption)
ax.xaxis.label.set_size(8)              # try 8–10
ax.set_title("Global Feature Importance (SHAP)", fontsize=12, pad=10)

plt.legend(
    loc="upper center",
    bbox_to_anchor=(0.5, -0.2),   # move legend below chart
    ncol=5,                        # number of columns (adjust)
    fontsize="small",
    frameon=False
)
#plt.tight_layout()


# After all plotting calls
out_path = OUTPUT_DIR / "shap_importance.png"   # or .pdf/.svg for vector (best for print)
plt.savefig(out_path, dpi=300, bbox_inches='tight')   # 300–600 dpi for print

plt.show()

### Random Forest

In [ ]:
# Initialize Random Forest
# class_weight='balanced' adds an extra layer of protection alongside SMOTE
from sklearn.ensemble import RandomForestClassifier
results=[]
rf_clf = RandomForestClassifier(
    n_estimators=100,
    max_depth=None,
    min_samples_split=2,
    random_state=42,
    n_jobs=-1
)

res_rf_p = evaluate_model(
    rf_clf, X_train_p, X_test_p, y_train, y_test,
    model_name="Random Forest - Primary Only"
)
results.append(res_rf_p)

In [ ]:
# Initialize Random Forest
# class_weight='balanced' adds an extra layer of protection alongside SMOTE
rf_clf = RandomForestClassifier(
    n_estimators=100,
    max_depth=None,
    min_samples_split=2,
    random_state=42,
    n_jobs=-1
)

res_rf_m = evaluate_model(
    rf_clf, X_train_m, X_test_m, y_train, y_test,
    model_name="Random Forest - Mutations Only"
)
results.append(res_rf_m)

In [ ]:
# Initialize Random Forest
# class_weight='balanced' adds an extra layer of protection alongside SMOTE
rf_clf = RandomForestClassifier(
    n_estimators=100,
    max_depth=None,
    min_samples_split=2,
    random_state=42,
    n_jobs=-1
)

res_rf_c = evaluate_model(
    rf_clf, X_train_c, X_test_c, y_train, y_test,
    model_name="Random Forest - Combined"
)
results.append(res_rf_c)

In [ ]:
df_results = pd.DataFrame(results)

df_results.plot(x="Model", y=["Accuracy", "MacroF1", "WeightedF1"],
                kind="bar", figsize=(8,6))
plt.title("Performance Comparison Across Feature Sets")
plt.ylabel("Score")
plt.xticks(rotation=0)
plt.ylim(0, 1)  # metrics are between 0 and 1
plt.show()

### SVM

In [ ]:
from sklearn.svm import SVC
results=[]
svm_clf = SVC(
    kernel='linear',
    C=1.0,
    probability=True,
    random_state=42
)

res_svm_p = evaluate_model(
    svm_clf, X_train_p, X_test_p, y_train, y_test,
    model_name="SVM - Primary Only"
)
results.append(res_svm_p)

In [ ]:
svm_clf = SVC(
    kernel='linear',
    C=1.0,
    probability=True,
    random_state=42
)

res_svm_m = evaluate_model(
    svm_clf, X_train_m, X_test_m, y_train, y_test,
    model_name="SVM - Mutations Only"
)
results.append(res_svm_m)

In [ ]:
svm_clf = SVC(
    kernel='linear',
    C=1.0,
    probability=True,
    random_state=42
)

res_svm_c = evaluate_model(
    svm_clf, X_train_c, X_test_c, y_train, y_test,
    model_name="SVM - Combined"
)
results.append(res_svm_c)

In [ ]:
df_results = pd.DataFrame(results)

df_results.plot(x="Model", y=["Accuracy", "MacroF1", "WeightedF1"],
                kind="bar", figsize=(8,6))
plt.title("Performance Comparison Across Feature Sets")
plt.ylabel("Score")
plt.xticks(rotation=0)
plt.ylim(0, 1)  # metrics are between 0 and 1
plt.show()

### MLP

In [ ]:
from sklearn.neural_network import MLPClassifier
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np
import torch

# 1. Get the unique classes as they appear in your LabelEncoder
# This ensures the weights match the order of the model's output layer
unique_classes = np.unique(le.transform(y_train))

# 2. Calculate 'balanced' weights
# Formula: n_samples / (n_classes * np.bincount(y))
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=unique_classes,
    y=le.transform(y_train)
)

# 3. Convert to a PyTorch float tensor
weights_tensor = torch.tensor(class_weights, dtype=torch.float32)

print("Weights calculated for classes:", len(weights_tensor))

In [ ]:
class ClinicalGenomicMLP(nn.Module):
    def __init__(self, input_size, num_classes):
        super(ClinicalGenomicMLP, self).__init__()

        # Layer 1: High-capacity input processing
        self.layer1 = nn.Sequential(
            nn.Linear(input_size, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.3)
        )

        # Layer 2: Feature extraction
        self.layer2 = nn.Sequential(
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.2)
        )

        # Layer 3: Dimensionality reduction
        self.layer3 = nn.Sequential(
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.1)
        )

        # Output: Logits for the metastatic sites
        self.output = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        return self.output(x)

In [ ]:
# Convert DataFrames to Tensors
X_train_t = torch.tensor(X_train_c.values, dtype=torch.float32)
y_train_t = torch.tensor(le.transform(y_train), dtype=torch.long)

X_test_t = torch.tensor(X_test_c.values, dtype=torch.float32)
y_test_t = torch.tensor(le.transform(y_test), dtype=torch.long)

# DataLoader for batching
train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=64, shuffle=True)

# Initialize Model
model = ClinicalGenomicMLP(X_train_c.shape[1], len(le.classes_))

# LOSS FUNCTION: Include the weights here to handle imbalance
criterion = nn.CrossEntropyLoss(weight=weights_tensor)
optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.01)

In [ ]:
# 1. Lower the learning rate
optimizer = optim.AdamW(model.parameters(), lr=0.0005, weight_decay=0.01)

model.train()
for epoch in range(100):
    epoch_loss = 0
    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()

        # 2. ADD GRADIENT CLIPPING HERE
        # max_norm=1.0 is a standard starting point
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()
        epoch_loss += loss.item()

    if epoch % 10 == 0:
        print(f"Epoch {epoch}, Average Loss: {epoch_loss/len(train_loader):.4f}")

In [ ]:

# Final Evaluation
model.eval()
with torch.no_grad():
    logits = model(X_test_t)
    _, predicted = torch.max(logits, 1)
    y_pred = le.inverse_transform(predicted.numpy())

# Evaluation Report
from sklearn.metrics import classification_report, f1_score
print(f"Macro F1 Score: {f1_score(y_test, y_pred, average='macro'):.4f}")
print(classification_report(y_test, y_pred))

### TabTransformer

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder

# ------------------------
# 1. Load data (replace with your files)
# ------------------------
clinical = pd.read_csv(CLINICAL_TSV, sep="\t")   # MSK clinical
clinical = clinical.dropna(subset=["Metastatic Site"])
mutations = pd.read_csv(MUTATIONS_MAF, sep="\t", comment='#', low_memory=False)


In [ ]:
# Keep only needed columns
clinical = clinical[["Sample ID", "Primary Tumor Site", "Metastatic Site"]]
mutations = mutations[["Hugo_Symbol", "Tumor_Sample_Barcode"]]

clinical = clinical.rename(columns={'Sample ID': 'Tumor_Sample_Barcode'})

trmerged = clinical.merge(mutations,
                  how='inner',
                  on='Tumor_Sample_Barcode')


mut_per_patient = (
    trmerged.groupby(["Tumor_Sample_Barcode", "Primary Tumor Site", "Metastatic Site"])["Hugo_Symbol"]
      .apply(lambda x: list(set(x)))  # unique mutations per patient
      .reset_index()
)

# Step 2: choose top-k mutation slots per patient (e.g., k=20)
k = 20
mut_per_patient["Mut_Slots"] = mut_per_patient["Hugo_Symbol"].apply(
    lambda muts: muts[:k] + ["None"]*(k - len(muts)) if len(muts) < k else muts[:k]
)

# Step 3: expand into columns MUT_1, MUT_2, ..., MUT_k
mut_expanded = pd.DataFrame(mut_per_patient["Mut_Slots"].to_list(), columns=[f"MUT_{i+1}" for i in range(k)])

# Step 4: final dataset
df_wide = pd.concat([mut_per_patient.drop(columns=["Hugo_Symbol", "Mut_Slots"]), mut_expanded], axis=1)

df_wide.head()

In [ ]:
from sklearn.preprocessing import LabelEncoder
import pandas as pd

# Define columns
categorical_cols = ["Primary Tumor Site"] + [f"MUT_{i+1}" for i in range(k)]
target_col = "Metastatic Site"

# -----------------------
# Step 1. Merge rare classes into "Other"
# -----------------------
min_count = 5  # threshold for rare classes
value_counts = df_wide[target_col].value_counts()

# Replace rare classes
df_wide[target_col] = df_wide[target_col].apply(
    lambda x: x if value_counts[x] >= min_count else "Other"
)

# -----------------------
# Step 2. Encode categorical + target columns
# -----------------------
encoders = {}
for col in categorical_cols + [target_col]:
    le = LabelEncoder()
    df_wide[col] = le.fit_transform(df_wide[col].astype(str))
    encoders[col] = le

# -----------------------
# Step 3. Build matrices
# -----------------------
X = df_wide[categorical_cols].values
y = df_wide[target_col].values

In [ ]:
from sklearn.model_selection import train_test_split
import torch

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Convert to torch tensors
X_train = torch.tensor(X_train, dtype=torch.long)
X_test  = torch.tensor(X_test, dtype=torch.long)
y_train = torch.tensor(y_train, dtype=torch.long)
y_test  = torch.tensor(y_test, dtype=torch.long)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

In [ ]:
import torch
import torch.nn as nn

# Number of categories per feature
categories = [len(encoders[col].classes_) for col in categorical_cols]
num_classes = len(encoders[target_col].classes_)

print("Categories per feature:", categories)
print("Number of classes:", num_classes)

class TabTransformer(nn.Module):
    def __init__(self, categories, num_classes, dim=32, depth=6, heads=8, mlp_hidden=64):
        super().__init__()
        # Embeddings per categorical feature
        self.embeddings = nn.ModuleList([
            nn.Embedding(cat_size, dim) for cat_size in categories
        ])
        # Transformer encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=dim, nhead=heads, dim_feedforward=mlp_hidden, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=depth)
        # Classifier head
        self.fc = nn.Sequential(
            nn.Linear(dim * len(categories), 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, num_classes)
        )
    def forward(self, x_categ):
        embs = [emb(x_categ[:, i]) for i, emb in enumerate(self.embeddings)]
        x = torch.stack(embs, dim=1)      # (batch, n_features, dim)
        x = self.transformer(x)           # transformer output
        x = x.flatten(1)                  # flatten all features
        return self.fc(x)

In [ ]:
from sklearn.metrics import accuracy_score, f1_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = TabTransformer(categories=categories, num_classes=num_classes).to(device)


criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# Move data to GPU if available
X_train, y_train = X_train.to(device), y_train.to(device)
X_test, y_test   = X_test.to(device), y_test.to(device)

epochs = 80
for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()

    outputs = model(X_train)
    loss = criterion(outputs, y_train)
    loss.backward()
    optimizer.step()

    preds = outputs.argmax(dim=1)
    acc = accuracy_score(y_train.cpu(), preds.cpu())

    print(f"Epoch {epoch+1}/{epochs} | Loss: {loss.item():.4f} | Train Acc: {acc:.4f}")

In [ ]:
model.eval()
with torch.no_grad():
    test_outputs = model(X_test)
    test_preds = test_outputs.argmax(dim=1)

test_acc = accuracy_score(y_test.cpu(), test_preds.cpu())
test_f1 = f1_score(y_test.cpu(), test_preds.cpu(), average="weighted")

print("\n=== Test Results ===")
print(f"Accuracy: {test_acc:.4f}")
print(f"Weighted F1: {test_f1:.4f}")

#### Embedding

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split

# -----------------------
# Step 1. Extract features
# -----------------------
primary_site = df_wide["Primary Tumor Site"].values
mut_features = df_wide[[f"MUT_{i+1}" for i in range(k)]].values  # binary mutation features
y = df_wide["Metastatic Site"].values

# Train/test split
site_train, site_test, mut_train, mut_test, y_train, y_test = train_test_split(
    primary_site, mut_features, y, test_size=0.2, random_state=42, stratify=y
)

# Convert to torch tensors
site_train = torch.tensor(site_train, dtype=torch.long)
mut_train  = torch.tensor(mut_train, dtype=torch.float32)
y_train    = torch.tensor(y_train, dtype=torch.long)

site_test = torch.tensor(site_test, dtype=torch.long)
mut_test  = torch.tensor(mut_test, dtype=torch.float32)
y_test    = torch.tensor(y_test, dtype=torch.long)

train_ds = TensorDataset(site_train, mut_train, y_train)
test_ds  = TensorDataset(site_test, mut_test, y_test)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_ds, batch_size=64)

In [ ]:
class OrganotropismNet(nn.Module):
    def __init__(self, n_sites, n_numeric, embedding_dim=8, hidden_dim=64, n_classes=5):
        super().__init__()

        # Embedding for primary tumor site
        self.site_emb = nn.Embedding(n_sites, embedding_dim)

        # Fully connected layers
        input_dim = embedding_dim + n_numeric
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim//2)
        self.out = nn.Linear(hidden_dim//2, n_classes)

    def forward(self, site, numeric):
        site_vec = self.site_emb(site)  # (batch, embedding_dim)
        x = torch.cat([site_vec, numeric], dim=1)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.out(x)

In [ ]:
n_sites = len(encoders["Primary Tumor Site"].classes_)
n_numeric = mut_train.shape[1]
n_classes = len(encoders["Metastatic Site"].classes_)

model = OrganotropismNet(n_sites=n_sites, n_numeric=n_numeric, n_classes=n_classes)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(50):
    model.train()
    total_loss = 0
    for site_batch, mut_batch, y_batch in train_loader:
        optimizer.zero_grad()
        logits = model(site_batch, mut_batch)
        loss = criterion(logits, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}: Loss = {total_loss/len(train_loader):.4f}")

In [ ]:
from sklearn.metrics import classification_report

model.eval()
y_true, y_pred = [], []
with torch.no_grad():
    for site_batch, mut_batch, y_batch in test_loader:
        logits = model(site_batch, mut_batch)
        preds = torch.argmax(logits, dim=1)
        y_true.extend(y_batch.numpy())
        y_pred.extend(preds.numpy())

print(classification_report(y_true, y_pred, target_names=encoders["Metastatic Site"].classes_))

### Adding SMOTE

In [ ]:
# NOTE: SMOTE is applied to the training split only; the test set stays untouched.
# 1. Clean the target first
data = merged.dropna(subset=['Metastatic Site']).copy()

# 2. Merge rare classes (Threshold = 20)
counts = data['Metastatic Site'].value_counts()
rare_sites = counts[counts < 20].index
data['Metastatic Site'] = data['Metastatic Site'].replace(rare_sites, 'Other')

# 3. Define y and X from the SAME 'data' object
y = data['Metastatic Site']
# Ensure your features are updated based on the same 'data' rows
X_combined = pd.concat([X_primary, X_mut], axis=1).loc[data.index]

# 4. Split
X_train, X_test, y_train, y_test = train_test_split(
    X_combined, y, test_size=0.2, random_state=42, stratify=y
)

# 5. Verify - You should see NO classes with counts less than ~16
print("Verification of y_train counts:")
print(y_train.value_counts())

In [ ]:
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import LabelEncoder

# Initialize SMOTE
# k_neighbors=5 is standard; if a class has very few samples, you might need to lower this.
smote = SMOTE(random_state=42)

# We need to encode y_train numerically for SMOTE to process it
le_smote = LabelEncoder()
y_train_encoded = le_smote.fit_transform(y_train)

# --- Apply SMOTE to each feature set ---

# 1. Primary Site Only
X_train_p_res, y_train_p_res = smote.fit_resample(X_train_p, y_train_encoded)
# Decode y back to strings so your evaluate_model function still works as expected
y_train_p_res = le_smote.inverse_transform(y_train_p_res)

# 2. Genetic Mutations Only
X_train_m_res, y_train_m_res = smote.fit_resample(X_train_m, y_train_encoded)
y_train_m_res = le_smote.inverse_transform(y_train_m_res)

# 3. Combined Features
X_train_c_res, y_train_c_res = smote.fit_resample(X_train_c, y_train_encoded)
y_train_c_res = le_smote.inverse_transform(y_train_c_res)

print(f"Original training size: {len(y_train)}")
print(f"Resampled training size (Combined): {len(y_train_c_res)}")
print("New class distribution:\n", pd.Series(y_train_c_res).value_counts())

In [ ]:
results_smote = []

# Combined Model with SMOTE
clf = LogisticRegression(max_iter=500, solver='saga', n_jobs=-1)
res_smote = evaluate_model(clf, X_train_c_res, X_test_c, y_train_c_res, y_test, "Logistic Regression - Combined + SMOTE")
results_smote.append(res_smote)

In [ ]:
res_xgb_smote = evaluate_model(
    XGBClassifier(
        objective="multi:softmax",
        num_class=len(np.unique(y_train)),
        eval_metric="mlogloss",
        random_state=42
    ),
    X_train_c_res, X_test_c, y_train_c_res, y_test,
    model_name="XGBoost - Combined + SMOTE"
)

results_smote.append(res_xgb_smote)


In [ ]:
rf_clf = RandomForestClassifier(
    n_estimators=100,
    max_depth=None,
    min_samples_split=2,
    random_state=42,
    n_jobs=-1
)

res_rf_c_smote = evaluate_model(
    rf_clf, X_train_c_res, X_test_c, y_train_c_res, y_test,
    model_name="Random Forest - Combined + SMOTE"
)
results_smote.append(res_rf_c_smote)

In [ ]:
# Initialize SVM
# probability=True is needed if you plan to calculate AUC later
svm_clf = SVC(
    kernel='linear',
    C=1.0,
    probability=True,
    random_state=42
)

res_svm = evaluate_model(
    svm_clf, X_train_c_res, X_test_c, y_train_c_res, y_test,
    model_name="SVM - Combined + SMOTE"
)
results_smote.append(res_svm)


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

class ComplexMLP(nn.Module):
    def __init__(self, input_size, num_classes):
        super(ComplexMLP, self).__init__()

        # Layer 1: Input to 512
        self.layer1 = nn.Sequential(
            nn.Linear(input_size, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.3)
        )

        # Layer 2: 512 to 256
        self.layer2 = nn.Sequential(
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.2)
        )

        # Layer 3: 256 to 128
        self.layer3 = nn.Sequential(
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.1)
        )

        # Output Layer
        self.output = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        return self.output(x)

In [ ]:
# Prepare data
X_train_t = torch.tensor(X_train_c_res.values, dtype=torch.float32)
y_train_t = torch.tensor(le_smote.transform(y_train_c_res), dtype=torch.long)

X_test_t = torch.tensor(X_test_c.values, dtype=torch.float32)
y_test_t = torch.tensor(le_smote.transform(y_test), dtype=torch.long)

# Create DataLoader
train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=64, shuffle=True)

# Initialize model, loss, and optimizer
input_dim = X_train_c_res.shape[1]
num_out = len(le_smote.classes_)
model = ComplexMLP(input_dim, num_out)

criterion = nn.CrossEntropyLoss() # Standard for multiclass
optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.01) # AdamW is better for regularization

In [ ]:
model.train()
for epoch in range(100):
    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()

    if epoch % 10 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}")

In [ ]:
model.eval()
with torch.no_grad():
    test_outputs = model(X_test_t)
    _, predicted = torch.max(test_outputs, 1)
    y_pred = le_smote.inverse_transform(predicted.numpy())

# Now you can use your existing classification_report logic
from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred))

In [ ]:
df_results = pd.DataFrame(results_smote)

df_results.plot(x="Model", y=["Accuracy", "MacroF1", "WeightedF1"],
                kind="bar", figsize=(8,6))
plt.title("Performance Comparison Across Feature Sets")
plt.ylabel("Score")
plt.xticks(rotation=0)
plt.ylim(0, 1)  # metrics are between 0 and 1
plt.show()

## Reversing Y and X

### Importing and Merging

In [ ]:
import pandas as pd
df=pd.read_csv(CLINICAL_TSV, sep="\t")
df.head()

# === 1. Load the mutation file (MAF format) ===
# Adjust path to where you saved data_mutations.txt
mutations = pd.read_csv(MUTATIONS_MAF, sep="\t", comment='#', low_memory=False)

print("File shape:", mutations.shape)
print("Columns:", mutations.columns.tolist())
print(mutations.head())

# === 2. Inspect key fields ===
# Usually, 'Hugo_Symbol' = gene, 'Tumor_Sample_Barcode' = sample ID
print(mutations[['Hugo_Symbol', 'Tumor_Sample_Barcode', 'Variant_Classification']].head())

# === 3. Create a binary mutation matrix ===
# Pivot so each gene is a column, each sample is a row
# Mark 1 if mutated, 0 otherwise
binary_matrix = (
    mutations
    .assign(value=1)
    .pivot_table(index='Tumor_Sample_Barcode',
                 columns='Hugo_Symbol',
                 values='value',
                 aggfunc='max',
                 fill_value=0)
)
#Building a logistic regression model
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

# Drop samples without metastatic site annotation
data = merged.dropna(subset=['Metastatic Site']).copy()




### Pre-processing

In [ ]:
# Define X
X = data['Metastatic Site']

print("Number of metastatic cases:", len(X))
print("Number of classes:", X.nunique())
print("Class distribution:\n", X.value_counts())

# Plot a count bar chart for X
plt.figure(figsize=(12, 8))
X.value_counts().plot(kind='bar')
plt.title('Count of Metastatic Sites')
plt.xlabel('Metastatic Site')
plt.ylabel('Count')
plt.xticks(rotation=90)
plt.show()

In [ ]:
y = data['Primary Tumor Site']
print(y.isna().sum())
print("Number of classes:", y.nunique())
print("Class distribution:\n", y.value_counts().to_string())

# Plot a count bar chart for X
plt.figure(figsize=(12, 8))
y.value_counts().plot(kind='bar')
plt.title('Count of Metastatic Sites')
plt.xlabel('Metastatic Site')
plt.ylabel('Count')
plt.xticks(rotation=90)
plt.show()

In [ ]:
# Define the labels to exclude
exclude_labels = ['Cancer of Unknown Primary', 'Unknown']

# Use the tilde (~) to keep rows NOT in that list
data = data[~data['Primary Tumor Site'].isin(exclude_labels)].copy()

# Reset index to avoid alignment issues later
data = data.reset_index(drop=True)

In [ ]:
y = data['Primary Tumor Site']
print(y.isna().sum())
print("Number of classes:", y.nunique())
print("Class distribution:\n", y.value_counts().to_string())

# Plot a count bar chart for X
plt.figure(figsize=(12, 8))
y.value_counts().plot(kind='bar')
plt.title('Count of Metastatic Sites')
plt.xlabel('Metastatic Site')
plt.ylabel('Count')
plt.xticks(rotation=90)
plt.show()

In [ ]:
# 1. Start fresh with your target column
# (Assuming 'data' is your filtered dataframe from the previous step)
y = data['Primary Tumor Site']

# 2. Identify the labels to keep (those with 20 or more samples)
counts = y.value_counts()
major_sites = counts[counts >= 20].index

# 3. Use .apply() to keep major sites and rename everything else to 'Other'
# This is much safer than .replace() when dealing with potential index mismatches
y_merged = y.apply(lambda x: x if x in major_sites else 'Other')

# 4. Final verification
print("Final Class Count:", y_merged.nunique())
print("\nSmallest classes (should all be >= 20):")
print(y_merged.value_counts().tail())

# Plot a count bar chart for X
plt.figure(figsize=(12, 8))
y_merged.value_counts().plot(kind='bar')
plt.title('Count of Metastatic Sites')
plt.xlabel('Metastatic Site')
plt.ylabel('Count')
plt.xticks(rotation=90)
plt.show()

In [ ]:
primary_encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
X_primary = primary_encoder.fit_transform(data[['Metastatic Site']])

# Automatically fetch feature names
primary_feature_names = primary_encoder.get_feature_names_out()
X_primary = pd.DataFrame(X_primary, columns=primary_feature_names, index=data.index)
gene_columns = [col for col in merged.columns if col not in df.columns]  # assuming df is clinical data
X_mut = data[gene_columns].copy()
X_mut = X_mut.drop(columns=['Tumor_Sample_Barcode'])
X_combined = pd.concat([X_primary, X_mut], axis=1)

In [ ]:
X_train_p, X_test_p, y_train, y_test = train_test_split(
    X_primary, y_merged, test_size=0.2, random_state=42, stratify=y_merged
)

X_train_m, X_test_m, _, _ = train_test_split(
    X_mut, y_merged, test_size=0.2, random_state=42, stratify=y_merged
)

X_train_c, X_test_c, _, _ = train_test_split(
    X_combined, y_merged, test_size=0.2, random_state=42, stratify=y_merged
)

### Logistic Regression (Multiclass)

In [ ]:
# evaluate_model() is defined in the forward section above; this section reuses it.
from sklearn.linear_model import LogisticRegression

results = []

clf = LogisticRegression(max_iter=500, solver='saga', n_jobs=-1)
results.append(evaluate_model(clf, X_train_p, X_test_p, y_train, y_test, "Metastatic Site Only"))

clf = LogisticRegression(max_iter=500, solver='saga', n_jobs=-1)
results.append(evaluate_model(clf, X_train_m, X_test_m, y_train, y_test, "Mutations Only"))

clf = LogisticRegression(max_iter=500, solver='saga', n_jobs=-1)
results.append(evaluate_model(clf, X_train_c, X_test_c, y_train, y_test, "Combined"))

In [ ]:
df_results = pd.DataFrame(results)

df_results.plot(x="Model", y=["Accuracy", "MacroF1", "WeightedF1"],
                kind="bar", figsize=(8,6))
plt.title("Performance Comparison Across Feature Sets")
plt.ylabel("Score")
plt.xticks(rotation=0)
plt.ylim(0, 1)  # metrics are between 0 and 1
plt.show()

### XGBoost

In [ ]:
from xgboost import XGBClassifier
results = []

res_primary = evaluate_model(
    XGBClassifier(
        objective="multi:softmax",
        num_class=len(np.unique(y_train)),
        eval_metric="mlogloss",
        random_state=42
    ),
    X_train_p, X_test_p, y_train, y_test,
    model_name="XGBoost - Metastatic Site Only"
)
results.append(res_primary)

res_mutations = evaluate_model(
    XGBClassifier(
        objective="multi:softmax",
        num_class=len(np.unique(y_train)),
        eval_metric="mlogloss",
        random_state=42
    ),
    X_train_m, X_test_m, y_train, y_test,
    model_name="XGBoost - Mutations Only"
)
results.append(res_mutations)

res_combined = evaluate_model(
    XGBClassifier(
        objective="multi:softmax",
        num_class=len(np.unique(y_train)),
        eval_metric="mlogloss",
        random_state=42
    ),
    X_train_c, X_test_c, y_train, y_test,
    model_name="XGBoost - Combined"
)
results.append(res_combined)

In [ ]:
df_results = pd.DataFrame(results)

df_results.plot(x="Model", y=["Accuracy", "MacroF1", "WeightedF1"],
                kind="bar", figsize=(8,6))
plt.title("Performance Comparison Across Feature Sets")
plt.ylabel("Score")
plt.xticks(rotation=0)
plt.ylim(0, 1)  # metrics are between 0 and 1
plt.show()

### SHAP

In [ ]:
import shap
import matplotlib.pyplot as plt
import os

model = res_combined["TrainedModel"]
le = res_combined["LabelEncoder"]

# Use the test set for SHAP
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test_c)

# Global importance (bar plot)
plt.figure(figsize=(14, 10))
shap.summary_plot(shap_values, X_test_c, plot_type="bar", class_names=le.classes_,max_display=20, show=False)
ax = plt.gca()
for lbl in ax.get_yticklabels():
    lbl.set_fontsize(7)      # or 6–8
    lbl.set_va('center')

ax = plt.gca()

# Shrink x-axis tick labels
ax.tick_params(axis='x', labelsize=8)   # try 6–9

# Shrink x-axis title (the long SHAP caption)
ax.xaxis.label.set_size(8)              # try 8–10
ax.set_title("Global Feature Importance (SHAP)", fontsize=12, pad=10)

plt.legend(
    loc="upper center",
    bbox_to_anchor=(0.5, -0.2),   # move legend below chart
    ncol=5,                        # number of columns (adjust)
    fontsize="small",
    frameon=False
)
#plt.tight_layout()


# After all plotting calls
out_path = OUTPUT_DIR / "shap_importance.png"   # or .pdf/.svg for vector (best for print)
plt.savefig(out_path, dpi=300, bbox_inches='tight')   # 300–600 dpi for print

plt.show()

### Random Forest

In [ ]:
# Initialize Random Forest
# class_weight='balanced' adds an extra layer of protection alongside SMOTE
from sklearn.ensemble import RandomForestClassifier
results=[]
rf_clf = RandomForestClassifier(
    n_estimators=100,
    max_depth=None,
    min_samples_split=2,
    random_state=42,
    n_jobs=-1
)

res_rf_p = evaluate_model(
    rf_clf, X_train_p, X_test_p, y_train, y_test,
    model_name="Random Forest - Primary Only"
)
results.append(res_rf_p)

In [ ]:
rf_clf = RandomForestClassifier(
    n_estimators=100,
    max_depth=None,
    min_samples_split=2,
    random_state=42,
    n_jobs=-1
)

res_rf_m = evaluate_model(
    rf_clf, X_train_m, X_test_m, y_train, y_test,
    model_name="Random Forest - Mutations Only"
)
results.append(res_rf_m)

In [ ]:
rf_clf = RandomForestClassifier(
    n_estimators=100,
    max_depth=None,
    min_samples_split=2,
    random_state=42,
    n_jobs=-1
)

res_rf_c = evaluate_model(
    rf_clf, X_train_c, X_test_c, y_train, y_test,
    model_name="Random Forest - Combined"
)
results.append(res_rf_c)

In [ ]:
df_results = pd.DataFrame(results)

df_results.plot(x="Model", y=["Accuracy", "MacroF1", "WeightedF1"],
                kind="bar", figsize=(8,6))
plt.title("Performance Comparison Across Feature Sets")
plt.ylabel("Score")
plt.xticks(rotation=0)
plt.ylim(0, 1)  # metrics are between 0 and 1
plt.show()

### SVM

In [ ]:
from sklearn.svm import SVC
results=[]
svm_clf = SVC(
    kernel='linear',
    C=1.0,
    probability=True,
    random_state=42
)

res_svm_p = evaluate_model(
    svm_clf, X_train_p, X_test_p, y_train, y_test,
    model_name="SVM - Primary Only"
)
results.append(res_svm_p)

In [ ]:
svm_clf = SVC(
    kernel='linear',
    C=1.0,
    probability=True,
    random_state=42
)

res_svm_m = evaluate_model(
    svm_clf, X_train_m, X_test_m, y_train, y_test,
    model_name="SVM - Mutations Only"
)
results.append(res_svm_m)

In [ ]:
svm_clf = SVC(
    kernel='linear',
    C=1.0,
    probability=True,
    random_state=42
)

res_svm_c = evaluate_model(
    svm_clf, X_train_c, X_test_c, y_train, y_test,
    model_name="SVM - Combined"
)
results.append(res_svm_c)

In [ ]:
df_results = pd.DataFrame(results)

df_results.plot(x="Model", y=["Accuracy", "MacroF1", "WeightedF1"],
                kind="bar", figsize=(8,6))
plt.title("Performance Comparison Across Feature Sets")
plt.ylabel("Score")
plt.xticks(rotation=0)
plt.ylim(0, 1)  # metrics are between 0 and 1
plt.show()

### Naive Bayes

In [ ]:
# --- 1. Identify Features ---
mutation_cols = [col for col in df_wide.columns if col.startswith('MUT_')]
metastatic_feature_col = "Metastatic Site"
target_col = "Primary Tumor Site"
categorical_cols = [metastatic_feature_col] + mutation_cols

# --- 2. Handle Imbalance (The Threshold) ---
counts = df_wide[target_col].value_counts()
major_sites = counts[counts >= 20].index

# IMPORTANT: Create the cleaned column directly in a copy of the dataframe
df_input = df_wide.copy()
df_input[target_col] = df_input[target_col].apply(lambda x: x if x in major_sites else 'Other')

# --- 3. Label Encoding ---
reverse_encoders = {}

# Now when we loop, df_input[target_col] is already merged/cleaned
for col in categorical_cols + [target_col]:
    le = LabelEncoder()
    df_input[col] = le.fit_transform(df_input[col].astype(str))
    reverse_encoders[col] = le

# --- 4. Final X and y ---
X_arr = df_input[categorical_cols].values
y_arr = df_input[target_col].values # This will now have no classes with count < 20

# --- 5. The Split ---
X_train_raw, X_test_raw, y_train_raw, y_test_raw = train_test_split(
    X_arr, y_arr, test_size=0.2, random_state=42, stratify=y_arr
)

print(f"Data ready. Input features: {X_arr.shape[1]} | Target classes: {len(reverse_encoders[target_col].classes_)}")

In [ ]:
from sklearn.naive_bayes import BernoulliNB
from sklearn.metrics import classification_report

# Initialize and Train
nb_classifier = BernoulliNB(alpha=1.0) # Laplace smoothing enabled
nb_classifier.fit(X_train_raw, y_train_raw)

# Predict
nb_preds = nb_classifier.predict(X_test_raw)

# Results
print("=== BASELINE MODEL: Bernoulli Naive Bayes ===")
print(classification_report(
    y_test_raw,
    nb_preds,
    target_names=reverse_encoders[target_col].classes_
))

### MLP

In [ ]:
from sklearn.neural_network import MLPClassifier
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np
import torch
from sklearn.preprocessing import LabelEncoder

# 1. Initialize a NEW LabelEncoder for the merged classes
le_primary = LabelEncoder()

# 2. Fit and transform the training labels
# This ensures 'Other', 'Lung', 'Breast', etc., are mapped to 0, 1, 2... correctly
y_train_encoded = le_primary.fit_transform(y_train_raw)

# 3. Calculate 'balanced' weights using the NEW encoded labels
unique_classes = np.unique(y_train_encoded)
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=unique_classes,
    y=y_train_encoded
)

# 4. Convert to a PyTorch float tensor
weights_tensor = torch.tensor(class_weights, dtype=torch.float32)

print(f"Success! Weights calculated for {len(weights_tensor)} classes.")
print("Example mappings:", dict(zip(le_primary.classes_[:5], range(5))))

In [ ]:
class ClinicalGenomicMLP(nn.Module):
    def __init__(self, input_size, num_classes):
        super(ClinicalGenomicMLP, self).__init__()

        # Layer 1: High-capacity input processing
        self.layer1 = nn.Sequential(
            nn.Linear(input_size, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.3)
        )

        # Layer 2: Feature extraction
        self.layer2 = nn.Sequential(
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.2)
        )

        # Layer 3: Dimensionality reduction
        self.layer3 = nn.Sequential(
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.1)
        )

        # Output: Logits for the metastatic sites
        self.output = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        return self.output(x)

In [ ]:
from sklearn.preprocessing import LabelEncoder

# 1. Initialize and Fit the encoder on the cleaned labels
# Use 'y_merged' or whatever you named your final, filtered target series
le = LabelEncoder()
le.fit(y_merged)

# 2. Convert DataFrames/Series to Tensors using the synced encoder
X_train_t = torch.tensor(X_train_c.values, dtype=torch.float32)
# Use the freshly fitted encoder here
y_train_t = torch.tensor(le.transform(y_train), dtype=torch.long)

X_test_t = torch.tensor(X_test_c.values, dtype=torch.float32)
y_test_t = torch.tensor(le.transform(y_test), dtype=torch.long)

# 3. Re-initialize the Model with the updated number of classes
# len(le.classes_) should now match your output layer dimension
model = ClinicalGenomicMLP(X_train_c.shape[1], len(le.classes_))

# 4. DataLoader for batching
train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=64, shuffle=True)

# 5. Define Loss with Weights (ensure weights_tensor was calculated after the merge)
criterion = nn.CrossEntropyLoss(weight=weights_tensor.to(device))
optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.01)

print(f"Model initialized for {len(le.classes_)} classes.")

In [ ]:
# 1. Get the definitive number of classes from your LabelEncoder
num_classes = len(le.classes_)
print(f"Definitive number of classes: {num_classes}")

# 2. Re-calculate weights based on the ACTUAL training labels used
from sklearn.utils.class_weight import compute_class_weight

# Ensure we only calculate weights for classes actually present in y_train
unique_train_classes = np.unique(y_train_t.numpy())
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=unique_train_classes,
    y=y_train_t.numpy()
)

# 3. Create the weights tensor and move to device
weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)
print(f"Weights tensor shape: {weights_tensor.shape}") # This should now be [35]

# 4. Re-initialize the model with the CORRECT output dimension
# input_size should match your X_train features (e.g., 342)
model = ClinicalGenomicMLP(input_size=X_train_t.shape[1], num_classes=num_classes).to(device)

# 5. Re-define the criterion with the synced weights
criterion = nn.CrossEntropyLoss(weight=weights_tensor)

# 6. Re-define the optimizer for the new model parameters
optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.01)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Ensure your model and weights are on the device
model.to(device)
weights_tensor = weights_tensor.to(device)

epochs = 100
for epoch in range(epochs):
    model.train()
    total_loss = 0

    for batch_X, batch_y in train_loader:
        # --- THE FIX: MOVE DATA TO DEVICE ---
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        # ------------------------------------

        optimizer.zero_grad()
        outputs = model(batch_X)

        # Now outputs, batch_y, and weights_tensor are all on CUDA
        loss = criterion(outputs, batch_y)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    if (epoch + 1) % 10 == 0:
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {total_loss/len(train_loader):.4f}")

In [ ]:
# 1. Ensure model is in eval mode and on device
model.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

with torch.no_grad():
    # 2. THE FIX: Move X_test_t to the GPU
    X_test_device = X_test_t.to(device)

    # 3. Get logits from model
    logits = model(X_test_device)

    # 4. Get predictions and move them back to CPU for Scikit-Learn
    _, predicted = torch.max(logits, 1)
    y_pred_indices = predicted.cpu().numpy()

    # 5. Inverse transform to get string labels
    y_pred = le.inverse_transform(y_pred_indices)

# Evaluation Report
from sklearn.metrics import classification_report, f1_score
print(f"Macro F1 Score: {f1_score(y_test, y_pred, average='macro'):.4f}")
print(classification_report(y_test, y_pred))

### TabTransformer

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder

# ------------------------
# 1. Load data (replace with your files)
# ------------------------
clinical = pd.read_csv(CLINICAL_TSV, sep="\t")   # MSK clinical
clinical = clinical.dropna(subset=["Metastatic Site"])
mutations = pd.read_csv(MUTATIONS_MAF, sep="\t", comment='#', low_memory=False)


In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

# --- NEW REVERSED DEFINITIONS ---
# Target is now Primary, Input includes Metastatic Site
target_col = "Primary Tumor Site"
metastatic_feature_col = "Metastatic Site"
mutation_cols = [f"MUT_{i+1}" for i in range(k)]
categorical_cols = [metastatic_feature_col] + mutation_cols

# 1. Merge rare Primary Sites into "Other" (Higher threshold for Primary)
min_count = 20
value_counts = df_wide[target_col].value_counts()
df_wide[target_col] = df_wide[target_col].apply(
    lambda x: x if value_counts[x] >= min_count else "Other"
)

# 2. Encode all columns
encoders = {}
for col in categorical_cols + [target_col]:
    le = LabelEncoder()
    df_wide[col] = le.fit_transform(df_wide[col].astype(str))
    encoders[col] = le

# 3. Build Matrices
X = df_wide[categorical_cols].values
y = df_wide[target_col].values

# 4. Split with Stratification on the new target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Calculate Class Weights for the Primary Tumor Sites
from sklearn.utils.class_weight import compute_class_weight
weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
weights_tensor = torch.tensor(weights, dtype=torch.float32)

# Convert to Tensors
X_train = torch.tensor(X_train, dtype=torch.long)
X_test = torch.tensor(X_test, dtype=torch.long)
y_train = torch.tensor(y_train, dtype=torch.long)
y_test = torch.tensor(y_test, dtype=torch.long)

In [ ]:
categories = [len(encoders[col].classes_) for col in categorical_cols]
num_classes = len(encoders[target_col].classes_)

class TabTransformer(nn.Module):
    def __init__(self, categories, num_classes, dim=32, depth=6, heads=8, mlp_hidden=64):
        super().__init__()
        self.embeddings = nn.ModuleList([
            nn.Embedding(cat_size, dim) for cat_size in categories
        ])
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=dim, nhead=heads, dim_feedforward=mlp_hidden, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=depth)
        self.fc = nn.Sequential(
            nn.Linear(dim * len(categories), 128),
            nn.ReLU(),
            nn.Dropout(0.3), # Increased dropout for complex primary labels
            nn.Linear(128, num_classes)
        )

    def forward(self, x_categ):
        embs = [emb(x_categ[:, i]) for i, emb in enumerate(self.embeddings)]
        x = torch.stack(embs, dim=1)
        x = self.transformer(x)
        x = x.flatten(1)
        return self.fc(x)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = TabTransformer(categories=categories, num_classes=num_classes).to(device)
weights_tensor = weights_tensor.to(device)

# Use weighted loss to handle the 62+ possible primary sites
criterion = nn.CrossEntropyLoss(weight=weights_tensor)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.01)

In [ ]:
# Training loop (using your structure)
epochs = 200
model.train()
for epoch in range(epochs):
    optimizer.zero_grad()
    outputs = model(X_train.to(device))
    loss = criterion(outputs, y_train.to(device))
    loss.backward()

    # Add gradient clipping to prevent spikes in the reverse path
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

    optimizer.step()
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/{epochs} | Loss: {loss.item():.4f}")

# Final Report
from sklearn.metrics import classification_report

model.eval()
with torch.no_grad():
    test_outputs = model(X_test.to(device))
    test_preds = test_outputs.argmax(dim=1).cpu().numpy()

print("\n=== REVERSE PREDICTION: Identifying Primary from Metastasis ===")
print(classification_report(
    y_test.cpu().numpy(),
    test_preds,
    target_names=encoders[target_col].classes_
))

### Adding SMOTE

In [ ]:
import pandas as pd
df=pd.read_csv(CLINICAL_TSV, sep="\t")
df.head()

# === 1. Load the mutation file (MAF format) ===
# Adjust path to where you saved data_mutations.txt
mutations = pd.read_csv(MUTATIONS_MAF, sep="\t", comment='#', low_memory=False)

print("File shape:", mutations.shape)
print("Columns:", mutations.columns.tolist())
print(mutations.head())

# === 2. Inspect key fields ===
# Usually, 'Hugo_Symbol' = gene, 'Tumor_Sample_Barcode' = sample ID
print(mutations[['Hugo_Symbol', 'Tumor_Sample_Barcode', 'Variant_Classification']].head())

# === 3. Create a binary mutation matrix ===
# Pivot so each gene is a column, each sample is a row
# Mark 1 if mutated, 0 otherwise
binary_matrix = (
    mutations
    .assign(value=1)
    .pivot_table(index='Tumor_Sample_Barcode',
                 columns='Hugo_Symbol',
                 values='value',
                 aggfunc='max',
                 fill_value=0)
)
#Building a logistic regression model
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

# Drop samples without metastatic site annotation
data = merged.dropna(subset=['Metastatic Site']).copy()

# Define the labels to exclude
exclude_labels = ['Cancer of Unknown Primary', 'Unknown']

# Use the tilde (~) to keep rows NOT in that list
data = data[~data['Primary Tumor Site'].isin(exclude_labels)].copy()

# Reset index to avoid alignment issues later
data = data.reset_index(drop=True)

X = data['Metastatic Site']

y = data['Primary Tumor Site']

# 2. Identify the labels to keep (those with 20 or more samples)
counts = y.value_counts()
major_sites = counts[counts >= 20].index

# 3. Use .apply() to keep major sites and rename everything else to 'Other'
# This is much safer than .replace() when dealing with potential index mismatches
y_merged = y.apply(lambda x: x if x in major_sites else 'Other')


primary_encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
X_primary = primary_encoder.fit_transform(data[['Metastatic Site']])

# Automatically fetch feature names
primary_feature_names = primary_encoder.get_feature_names_out()
X_primary = pd.DataFrame(X_primary, columns=primary_feature_names, index=data.index)
gene_columns = [col for col in merged.columns if col not in df.columns]  # assuming df is clinical data
X_mut = data[gene_columns].copy()
X_mut = X_mut.drop(columns=['Tumor_Sample_Barcode'])
X_combined = pd.concat([X_primary, X_mut], axis=1)
X_train_c, X_test_c, y_train, y_test = train_test_split(
    X_combined, y_merged, test_size=0.2, random_state=42, stratify=y_merged
)


In [ ]:
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import LabelEncoder

# Initialize SMOTE
# k_neighbors=5 is standard; if a class has very few samples, you might need to lower this.
smote = SMOTE(random_state=42)

# We need to encode y_train numerically for SMOTE to process it
le_smote = LabelEncoder()
y_train_encoded = le_smote.fit_transform(y_train)


# 3. Combined Features
X_train_c_res, y_train_c_res = smote.fit_resample(X_train_c, y_train_encoded)
y_train_c_res = le_smote.inverse_transform(y_train_c_res)

print(f"Original training size: {len(y_train)}")
print(f"Resampled training size (Combined): {len(y_train_c_res)}")
print("New class distribution:\n", pd.Series(y_train_c_res).value_counts())

In [ ]:

results_smote = []

# Combined Model with SMOTE
clf = LogisticRegression(max_iter=500, solver='saga', n_jobs=-1)
res_smote = evaluate_model(clf, X_train_c_res, X_test_c, y_train_c_res, y_test, "Logistic Regression - Combined + SMOTE")
results_smote.append(res_smote)


In [ ]:
res_xgb_smote = evaluate_model(
    XGBClassifier(
        objective="multi:softmax",
        num_class=len(np.unique(y_train)),
        eval_metric="mlogloss",
        random_state=42
    ),
    X_train_c_res, X_test_c, y_train_c_res, y_test,
    model_name="XGBoost - Combined + SMOTE"
)

results_smote.append(res_xgb_smote)


In [ ]:
rf_clf = RandomForestClassifier(
    n_estimators=100,
    max_depth=None,
    min_samples_split=2,
    random_state=42,
    n_jobs=-1
)

res_rf_c_smote = evaluate_model(
    rf_clf, X_train_c_res, X_test_c, y_train_c_res, y_test,
    model_name="Random Forest - Combined + SMOTE"
)
results_smote.append(res_rf_c_smote)


In [ ]:
# Initialize SVM
# probability=True is needed if you plan to calculate AUC later
svm_clf = SVC(
    kernel='linear',
    C=1.0,
    probability=True,
    random_state=42
)

res_svm = evaluate_model(
    svm_clf, X_train_c_res, X_test_c, y_train_c_res, y_test,
    model_name="SVM - Combined + SMOTE"
)
results_smote.append(res_svm)
